# Seeing Data — class notebook

**36104 Data Visualisation and Narratives · Class 1**

A chart is a claim about how the world is organised. In this notebook you will
make charts, let an AI assistant make charts, and — most importantly — verify
the claims both of you produce.

## How to work in this notebook

- Work in VS Code (or Jupyter) **with your AI assistant enabled**. Every
  exercise is written so that the markdown context plus the code scaffold give
  the assistant what it needs to draft a useful completion.
- You may accept, edit, or ignore any suggestion. What you may **not** do is
  submit code that fails its verification cell.
- A verification cell that fails is a *finding*, not a failure — read the
  error, work out whether the code or the claim is wrong, and fix it.
- Keep the **verification ladder** beside you:
  **1 TRACE** what data/transformation produced the chart →
  **2 CHECK** labels, scales, units, totals →
  **3 TEST** the pattern in another view →
  **4 BOUND** the claim with its limits →
  **5 DISCLOSE** what the tool contributed.

In [ ]:
# Setup — run this cell, no need to edit it.
# It builds the synthetic public-transport patronage dataset used across this course:
# monthly rider counts for six NSW regions and four transport modes, 2019-2025,
# with a seasonal cycle and a COVID-shaped shock. Teaching data, not real data.
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

REGIONS = ["Inner Sydney", "Western Sydney", "Northern Beaches",
           "Central Coast", "Newcastle", "Illawarra"]
MODES = ["Train", "Bus", "Ferry", "Light rail"]
BASE = {"Train": 1_000_000, "Bus": 700_000, "Ferry": 90_000, "Light rail": 120_000}
FACTOR = {"Inner Sydney": 1.3, "Western Sydney": 1.1, "Northern Beaches": 0.55,
          "Central Coast": 0.45, "Newcastle": 0.5, "Illawarra": 0.42}

rows = []
for region in REGIONS:
    for mode in MODES:
        base = BASE[mode] * FACTOR[region]
        for date in pd.date_range("2019-01-01", "2025-12-01", freq="MS"):
            season = 1 + 0.08 * math.sin((date.month - 1) / 12 * 2 * math.pi)
            covid = 1.0
            if pd.Timestamp("2020-03-01") <= date <= pd.Timestamp("2021-12-01"):
                covid = 0.35 + 0.3 * (date - pd.Timestamp("2020-03-01")).days / 640
            elif date > pd.Timestamp("2021-12-01"):
                covid = min(1.0, 0.65 + 0.35 * (date - pd.Timestamp("2021-12-01")).days / 1100)
            noise = random.gauss(1, 0.03)
            rows.append({"date": date, "region": region, "mode": mode,
                         "riders": int(base * season * covid * noise)})

transport = pd.DataFrame(rows)
print(f"{len(transport):,} rows")
transport.head()

## Exercise 1 — The summary is not the shape

Anscombe's quartet: four small datasets constructed so that their summary
statistics agree almost exactly, while their shapes disagree completely.

**Task.** Compute the summary statistics for each dataset, then draw the four
scatterplots. Watch the moment the numbers stop being enough.

*Copilot works well here if you write the docstring first, then pause.*

In [ ]:
# Anscombe's quartet — the classic values (Anscombe, 1973).
anscombe = {
    "I":   dict(x=[10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
                y=[8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II":  dict(x=[10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
                y=[9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": dict(x=[10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
                y=[7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV":  dict(x=[8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
                y=[6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}


def summarise(name: str) -> dict:
    """Return summary statistics for one Anscombe dataset.

    Given the dataset key ("I".."IV"), return a dict with:
      mean_x, mean_y  — means of x and y
      var_x, var_y    — sample variances (ddof=1)
      corr            — Pearson correlation between x and y
    Round every value to 2 decimal places.
    """
    # TODO: implement using numpy (np.mean, np.var with ddof=1, np.corrcoef)
    raise NotImplementedError


for name in anscombe:
    print(name, summarise(name))

In [ ]:
# Verification — the whole point of the quartet is that these agree.
for name in anscombe:
    s = summarise(name)
    assert abs(s["mean_x"] - 9.0) < 0.01, f"{name}: mean_x should be 9.0, got {s['mean_x']}"
    assert abs(s["mean_y"] - 7.5) < 0.06, f"{name}: mean_y should be ~7.50, got {s['mean_y']}"
    assert abs(s["corr"] - 0.82) < 0.01, f"{name}: corr should be ~0.82, got {s['corr']}"
print("All four datasets agree on the summary statistics. Now look at them.")

In [ ]:
# Now the picture. Draw the four datasets as a 2x2 grid of scatterplots,
# same axis limits on every panel, one regression line per panel.
def plot_quartet(anscombe: dict) -> None:
    """2x2 scatterplots of Anscombe's quartet with fitted lines.

    Same x/y limits everywhere (x: 2..20, y: 2..14) so the panels are
    directly comparable. Title each panel with its key.
    """
    # TODO: matplotlib subplots; np.polyfit(x, y, 1) for each fitted line
    raise NotImplementedError


plot_quartet(anscombe)

**Reflect** (edit this cell): which panel most changes your interpretation of
"correlation ≈ 0.82"? What claim would each panel justify — and what claim
would it forbid?

## Exercise 2 — Generate a chart, then interrogate it

Now the real dataset. The `transport` DataFrame holds monthly rider counts by
`region` and `mode`, 2019–2025, including a COVID-shaped collapse and recovery.

**Task.** Ask your assistant to draw *"total monthly riders per mode over
time"* — write the docstring below and let it draft the body. Then run the
verification cell, which climbs the first two rungs of the ladder (TRACE and
CHECK) for you. If verification fails, the bug is somewhere between the
assistant's idea of the data and the actual data. Find it.

In [ ]:
def plot_mode_recovery(transport: pd.DataFrame) -> pd.DataFrame:
    """Line chart of total monthly riders per mode, 2019-2025.

    - Aggregate riders across regions: one monthly total per mode.
    - One line per mode, labelled, with a legend and y-axis in millions.
    - Title the chart with the claim it supports, not a restatement of the axes.
    - RETURN the aggregated DataFrame used for plotting
      (columns: date, mode, riders) so it can be verified.
    """
    # TODO: groupby(["date", "mode"]), then plot one line per mode
    raise NotImplementedError


plotted = plot_mode_recovery(transport)
plotted.head()

In [ ]:
# Verification — TRACE and CHECK.
# 1 TRACE: the plotted table must reconcile with the source table.
assert plotted["riders"].sum() == transport["riders"].sum(), (
    "Aggregation lost or duplicated riders: plotted total != source total")
# 2 CHECK: every mode present, every month present, no NaNs.
assert set(plotted["mode"]) == set(MODES), "A mode went missing in the aggregation"
assert plotted.groupby("mode")["date"].nunique().eq(84).all(), (
    "Each mode should have 84 monthly points (2019-01..2025-12)")
assert plotted["riders"].notna().all(), "NaNs appeared during aggregation"
# Spot total, hand-derivable: Train riders in Jan 2019 across all six regions.
jan = pd.Timestamp("2019-01-01")
jan_train = plotted[(plotted["mode"] == "Train") & (plotted["date"] == jan)]["riders"].iloc[0]
source_jan_train = transport[(transport["mode"] == "Train") & (transport["date"] == jan)]["riders"].sum()
assert jan_train == source_jan_train, "Spot total disagrees for Train, Jan 2019"
print("TRACE ✓  CHECK ✓ — now do TEST yourself: does the recovery story survive a per-region view?")

In [ ]:
# 3 TEST — does the pattern survive another view?
# Draw the same measure faceted by region (small multiples). If the "recovery"
# claim holds overall but fails for a region, the overall chart compresses that.
def plot_recovery_by_region(transport: pd.DataFrame) -> None:
    """Small-multiple line charts: total riders per month, one panel per region.

    2x3 grid, shared y-axis in millions, panel titled by region.
    """
    # TODO
    raise NotImplementedError


plot_recovery_by_region(transport)

**Reflect** (edit this cell): name one claim the overall chart supports that
the per-region view weakens, and one limitation (rung 4, BOUND) you would
attach before publishing either chart.

## Exercise 3 — Audit a generated interpretation

Below is a fluent, confident, AI-style interpretation of the transport data.
Some of its claims are supported by the data you have; some are contradicted
by it; some *cannot be checked from this dataset at all*.

> "Public transport has fully recovered from the pandemic. Ridership across
> all modes now exceeds pre-COVID levels, driven primarily by the return of
> office workers to the Sydney CBD. Ferry patronage has been the most
> resilient mode throughout, and Western Sydney has overtaken Inner Sydney
> as the busiest region. The data shows that commuters strongly prefer rail
> over road-based transport."

**Task.** Classify each claim, then *prove* your classification for the
checkable ones with a query against `transport`.

- `"supported"` — the dataset agrees
- `"unsupported"` — the dataset disagrees
- `"unverifiable"` — this dataset cannot answer it either way

In [ ]:
claims = {
    "ridership across all modes now exceeds pre-COVID levels": "...",
    "the recovery is driven by office workers returning to the CBD": "...",
    "ferry patronage has been the most resilient mode throughout": "...",
    "Western Sydney has overtaken Inner Sydney as the busiest region": "...",
    "commuters strongly prefer rail over road-based transport": "...",
}

# TODO: replace each "..." with "supported", "unsupported", or "unverifiable".
# For each claim you mark supported/unsupported, add a query below that shows it.
# Example probe (compare 2025 with 2019, per mode):
# transport[transport["date"] >= pd.Timestamp("2025-01-01")].groupby("mode")["riders"].sum()

In [ ]:
# Verification — classification sanity check.
allowed = {"supported", "unsupported", "unverifiable"}
assert all(v in allowed for v in claims.values()), (
    "Every claim needs one of: supported / unsupported / unverifiable")

# The motive claim ("driven by office workers...") names a cause the dataset
# does not record; the preference claim infers psychology from counts.
for motive in ["the recovery is driven by office workers returning to the CBD",
               "commuters strongly prefer rail over road-based transport"]:
    assert claims[motive] == "unverifiable", (
        f"Re-read this claim: '{motive}' — does ANY column in transport "
        "record causes or preferences?")
print("Classification accepted. Bring your probe queries to the studio discussion.")

**The lesson.** The interpretation reads as insight. Two of its five claims
could never be checked from this data — yet nothing in the prose marks them as
different from the rest. That marking is *your* job, every time.

## Exercise 4 — Make a true chart lie (then fix it)

Every number in a chart can be correct and the impression still wrong. You will
now build two misleading-but-accurate charts of Ferry ridership, then the
honest one. Knowing how the trick is done is the best defence against it.

- **Lie 1, truncated axis** — crop the y-axis so the post-COVID recovery looks
  like an explosion.
- **Lie 2, cherry-picked window** — show only a window in which ferries appear
  to be in permanent decline.
- **Honest** — zero baseline, full 2019–2025 window, title stating the finding.

Each function must **return the Axes object** so the verification cell can
inspect what your chart actually asserts.

In [ ]:
# Given: monthly Ferry riders across all regions (run, don't edit).
ferry = (transport[transport["mode"] == "Ferry"]
         .groupby("date")["riders"].sum())
ferry.tail()

In [ ]:
def lie_truncated(ferry: pd.Series):
    """Line chart of 2024-2025 Ferry riders with a savagely cropped y-axis,
    so a modest recovery reads as a boom. Title it like a press release.
    RETURN the Axes."""
    # TODO
    raise NotImplementedError


def lie_window(ferry: pd.Series):
    """Line chart of Ferry riders over a window of at most 18 months chosen
    so the series appears to be declining. Title it pessimistically.
    RETURN the Axes."""
    # TODO
    raise NotImplementedError


def honest(ferry: pd.Series):
    """The fair chart: full 2019-2025 window, y-axis from zero, title that
    states the actual finding (collapse and partial recovery).
    RETURN the Axes."""
    # TODO
    raise NotImplementedError


ax1, ax2, ax3 = lie_truncated(ferry), lie_window(ferry), honest(ferry)

In [ ]:
# Verification — the axes must actually commit each sin, and the honest
# chart must actually be honest.
assert ax1.get_ylim()[0] > ferry.min() * 0.5, (
    "Lie 1 isn't truncated enough: the y-axis floor should sit well above zero")
span_days = ax2.get_xlim()[1] - ax2.get_xlim()[0]
assert span_days < 560, "Lie 2's window is too wide to be a cherry-pick (<18 months)"
assert ax3.get_ylim()[0] <= 0, "The honest chart must include the zero baseline"
honest_span = ax3.get_xlim()[1] - ax3.get_xlim()[0]
assert honest_span > 2200, "The honest chart must show the full 2019-2025 window"
print("Two effective lies and one honest chart. Screenshot all three for the studio wall.")

**Reflect** (edit this cell): which lie was harder to catch when you imagined
it in a news feed? What single sentence of disclosure would defuse each one?

## Exercise 5 (stretch) — Draw the uncertainty

The recovery chart from Exercise 2 shows one line per mode, each drawn with
total confidence. But the monthly values wobble. Show the wobble.

**Task.** For one mode, plot the 12-month rolling mean of riders with a shaded
band of ± one rolling standard deviation. The band is the honesty.

In [ ]:
def plot_with_uncertainty(transport: pd.DataFrame, mode: str = "Train"):
    """Rolling mean (12 months) of the mode's total riders with a
    +/- 1 rolling-std shaded band (ax.fill_between). RETURN the Axes."""
    # TODO
    raise NotImplementedError


ax_u = plot_with_uncertainty(transport)

In [ ]:
# Verification — a line and a band must both exist.
assert len(ax_u.lines) >= 1, "Where is the rolling-mean line?"
assert len(ax_u.collections) >= 1, "Where is the shaded uncertainty band (fill_between)?"
print("Uncertainty drawn rather than suppressed. Rung 4 of the ladder, made visible.")

## AI disclosure block

Before you close this notebook, complete the course disclosure block. This is
the same block you will attach to every assessed artefact.

```text
What did the intelligent tool contribute?
  (e.g. "generated the first draft of plot_mode_recovery"; "suggested the melt call")
How was each contribution checked?
  (e.g. "ran the verification cell"; "hand-computed the Inner Sydney 2024 total")
What did you write or decide yourself?
What would you not trust the tool to do in this notebook?
```

Write your answers in the cell below.

*Your disclosure:*

- **Tool contributed:** …
- **How checked:** …
- **I wrote/decided:** …
- **Would not trust:** …